### Soma total de vendas por dia, ordens com status de 'Delivered' e State = NY

In [0]:
%python
# Definir pastas do projeto em variáveis para facilitar
bronze_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/bronze/'
silver_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/'
gold_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/'
resource_path = 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/resource/origem'
resource_path_volume = '/Volumes/bikestore/logistics/bikestore_resource/origem/'

In [0]:
%python
display(dbutils.fs.ls(silver_path))

path,name,size,modificationTime
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/customer/,customer/,0,0
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/orders/,orders/,0,0
abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/silver/product/,product/,0,0


In [0]:
%python
# Criando várias tabelas temporárias 
silver_map = {
    'tmp_silver_customer': f'{silver_path}/customer/',
    'tmp_silver_orders': f'{silver_path}/orders/',
    'tmp_silver_product': f'{silver_path}/product/'
}
for view_name, path in silver_map.items():
    (spark.read.format('delta')
    .load(path)
    .createOrReplaceTempView(view_name)
) 

In [0]:
SELECT
shipped_date,
ROUND(SUM(total_sale), 2) AS total_sale
-- state,
-- status
FROM tmp_silver_orders
WHERE state = 'NY'
AND status = 'Delivered'
AND shipped_date IS NOT NULL
GROUP BY shipped_date
-- GROUP BY state, status

shipped_date,total_sale
2016-01-03,3217.95
2016-01-06,4143.05
2016-01-05,12980.05
2016-01-07,3155.95
2016-01-11,3278.05
2016-01-12,4030.05
2016-01-18,2083.16
2016-01-19,7977.27
2016-01-21,27263.69
2016-01-20,4205.96


In [0]:
DESCRIBE tmp_silver_orders

col_name,data_type,comment
order_id,int,null
customer_id,int,null
status,string,null
order_status,int,null
order_date,date,null
required_date,date,null
shipped_date,string,null
store_name,string,null
state,string,null
city,string,null


In [0]:
%python
# Salvando no formato delta na camada Gold
df_sales_ny_Gold = spark.sql("""
SELECT
shipped_date,
ROUND(SUM(total_sale), 2) AS total_sale
-- state,
-- status
FROM tmp_silver_orders
WHERE state = 'NY'
AND status = 'Delivered'
AND shipped_date IS NOT NULL
GROUP BY shipped_date
-- GROUP BY state, status
""")

# Salvar e transformar em formato delta na camada Gold
df_sales_ny_Gold.write \
    .mode('overwrite') \
    .format('delta') \
    .option('mergeSchema', 'true') \
    .save(f'{gold_path}/sales_ny')

In [0]:
%sql
/* Criar tabela física de produção, este comando precisa rodar só 1 vez, pois quando os arquivos forem atualizados a tabela refletirá o resultado (Vai obter o log mais recente e trazer dados do(s) arquivo(s) atualizado)*/
CREATE TABLE IF NOT EXISTS bikestore.logistics.gold_sales_ny
LOCATION 'abfss://unitycatalog-ext-azure@databricksextazure.dfs.core.windows.net/bikestore/gold/sales_ny';

In [0]:
SELECT * FROM bikestore.logistics.gold_sales_ny;

shipped_date,total_sale
2016-01-24,9021.5
2016-02-07,4076.49
2016-02-09,9025.54
2016-03-29,4945.14
2016-04-09,8696.07
2016-04-19,3930.12
2016-05-27,2465.45
2016-06-15,3163.04
2016-06-27,3810.82
2016-07-16,2687.14
